# Human Pose Estimation

Human pose estimation (HPE) is a markerless approach to motion capture (mocap) that uses computer vision (CV) and deep learning (DL) to detect humans from sensor data and localise their keypoints. This information can be converted into **quantitative biomechanical measurements**.
<figure style="text-align: center;">
    <img src="HPE_markerless.png" width="900" style="display: block; margin: 0 auto;"> 
</figure>

The gold standard for quantitative biomechanical assessment is still marker-based motion capture due to its reliability and precision. Marker-based mocap involves placing infrared-reflective (IR) markers on anatomical landmarks, which are then tracked by specialised IR cameras. This process does not require deep learning or neural networks. However, these systems are largely confined to specialised gait laboratories due to their high cost, complex setup, and operational demands. 

<figure style="text-align: center;">
    <img src="marker-based_mocap.jpg" width="900" style="display: block; margin: 0 auto;">
    <figcaption>Marker-based motion capture system</figcaption>
</figure>
HPE systems are increasingly being used to lower the barriers to quantitative biomechanical analysis and to support telehealth and remote patient monitoring (RPM) applications. To date, they have been most successful in use cases that require active patient engagement, such as point-in-time assessments and telerehabilitation for managing conditions such as Parkinson's diease, post-stroke impairments, musculoskeletal injuries. 

<figure style="text-align: center;">
<img src="leg_lat.gif" width="600">
</figure>

Current HPE systems are generally less effective for continual, unconstrained monitoring, where the environment and activities are less controlled. Depending on the method and sensing modality, limitations may include:
* reduced robustness to occlusion
* difficulty handling multiple people
* high computational demands / insufficient real-time performance
* limited 3D accuracy
* privacy concerns
* sensitivity to lighting or weather conditions.

These limitations make it harder to use HPE reliably for continuous monitoring which can support surveillance tasks (e.g. detecting threats or hazardous events such as falls) or tasks which benefit from long-term longitudinal measurement (e.g. fall-risk assessment / fall prevention), diagnosing conditions. They may also limit the extent to which such systems can reduce Hawthorne effects in real-world monitoring. 

<figure style="text-align: center;">
<img src="HPE_gait_tracks_.gif" width="900">
</figure>

In this lesson, we focus only on single-frame HPE methods. Temporal HPE methods, which use multiple frames and may involve sequence models such as recurrent neural networks, are outside the scope of this lesson.

## HPE Datasets

**Monocular RGB-camera-based 2D HPE** is the most widely studied and best-supported form of HPE. One major reason is that large and diverse image datasets can be collected relatively easily with internet search engines and then annotated manually with 2D keypoints. This has enabled rapid progress in model development, benchmarking, and transfer learning.

| Dataset | Keypoints | Samples | Poses | 
| -------- | -------- | -------- | -------- |
| [MS COCO](https://cocodataset.org/#home) | 17 | 200,000 | 250,000 |
| [COCO-Wholebody](https://www.ecva.net/papers/eccv_2020/papers_ECCV/papers/123540188.pdf) | 133 | 200,000 | 250,000 |
| [MPII](https://www.mpi-inf.mpg.de/departments/computer-vision-and-machine-learning/software-and-datasets/mpii-human-pose-dataset) | 16 | 25,000 | 40,000 |
| [AiC](https://github.com/AIChallenger/AI_Challenger_2017) | 14 | 300,000 | 700,000 |
| [J-HMDB](https://is.mpg.de/ps/en/projects/jhmdb) | 15 | 31,838 | 31,838 |
| [Crowdpose](https://github.com/jeffffffli/CrowdPose) | 13 | 20,000 | 80,000 |
| [PoseTrack18](https://doi.org/10.1109/CVPR.2018.00542) | 16 | 23,000 | 150,000 |
| [OCHuman](https://github.com/liruilong940607/OCHumanApi) | 17 | 4,730 | 8,110 |

Many datasets for **3D HPE** are collected in much more controlled environments. Participants are typically recorded while performing scripted activities, and 3D ground-truth pose can be obtained using marker-based mocap. These labels are highly reliable, but the need for mocap usually makes data collection much more costly and restrictive. As a result, these datasets often contain fewer participants, less environmental variation, and narrower activity coverage than large-scale monocular RGB 2D datasets. Some pose types or interaction scenarios may also be underrepresented if they are difficult to capture reliably with markers (e.g. lying down can occlude markers).

The **[Human3.6M](http://vision.imar.ro/human3.6m/description.php)** ([paper](https://doi.org/10.1109/TPAMI.2013.248)) dataset is a standard benchmark for monocular RGB-based single-person 3D human pose estimation. Data was captured from human participants using a synchronised multi-view RGB camera setup and a marker-based mocap system. While it is very large, involving 3.6 million human poses in total, it severly lacks variation because it was collected in a controlled lab environment involving only 11 participants performing scripted single-person activities. Additionally, adjacent frames within the recordings would be nearly identical. 

| MS COCO Samples | Human3.6M Samples |
| -------- | -------- |
| <img src="rgb_hpe_variation.png" width="450"> | <img src="human_3.6m_variation.jpeg" width="450"> | 

Datasets may also support 2D or 3D HPE through **cross-modal supervision**, where an existing pose-estimation system trained using another sensing modality or ground-truth source is used to generate labels for a target modality that lacks direct pose ground truth.

* The **[HuPR](https://doi.org/10.1109/WACV56688.2023.00567)** dataset supports single-person mmWave-radar-based 2D HPE by recording participants using a synchronised radar and monocular RGB camera. HRNet, trained on MS COCO, was applied to the RGB video to produce pose labels, which are then used a supervision for radar-based models.

<figure style="text-align: center;">
<img src="hupr_preds.png" width="400" align="center">
<figcaption>Pose ground truth and radar-based predictions overlayed on RGB and radar-heatmap images from the HuPR dataset</figcaption>
</figure>

* The **[RT-Pose](https://doi.org/10.1007/978-3-031-73036-8_7)** dataset supports mmWave-radar-based multi-person 3D HPE by recording participants using synchronised radar and two monocular RGB cameras. The cameras are calibrated so that their relative rotation and translation are known, forming a stereo RGB camera pair capable of depth perception. 3D poses can then be obtained by triangulating 2D poses estimated from both views.

<figure style="text-align: center;">
<img src="RT-Pose_annotation2.jpeg" width="400" align="center">
<figcaption>RT-Pose's 3D pose ground truth obtained via triangulating 2D pose estimates from a stereoscopic RGN camera system</figcaption>
</figure>

<figure style="text-align: center;">
<img src="RT-Pose_annotation.gif" width="900" align="center">
<figcaption>3D pose ground truth from RT-Pose</figcaption>
</figure>

* The **[mRI](https://sizhean.github.io/mri)** dataset supports mmWave-radar-based single-person 3D HPE by recording participants using synchronised radar, wearable inertial sensors, and two Microsoft Kinect RGB-depth (RGB-D) camera. The Kinect Body Tracker is a proprietary pose estimation model trained using a private dataset with marker-based mocap ground truth and its outputs are used as supervision for the dataset.

<figure style="text-align: center;">
<img src="mri_RGBD.gif" width="900" align="center">
<figcaption>Two RGB-D camera streams used in the mRI dataset to obtain 3D pose ground truth</figcaption>
</figure>

<figure style="text-align: center;">
<img src="mri_multimodal.gif" width="900" align="center">
<figcaption>3D pose ground truth and estimates from a variety of modalities in the mRI dataset</figcaption>
</figure>

* The **[MM-Fi](https://ntu-aiot-lab.github.io/mm-fi)** dataset supports wireless-sensing-based 3D single-person HPE by recording participants using synchronised WiFi, mmWave radar, LiDAR, and a stereoscopic RGB camera system. 3D pose labels are triangulated from 2D poses estimated with HRNet applied to each camera. 

<figure style="text-align: center;">
<img src="MMFi_multimodal.gif" width="900" align="center">
<figcaption>3D pose ground truth and estimates from a variety of modalities in the MM-Fi dataset.</figcaption>
</figure>

### Pose Topologies

These camera-based datasets define their own skeleton topologies which can limit interoperability between datasets and models. Some topologies used in popular monocular RGB 2D datasets are shown below. 

<figure style="text-align: center;">
<img src="pose_topologies.png" width="900" align="center">
</figure>

## Pose Estimates

The final output of a HPE system is a set of **pose estimates** describing the predicted location of human body keypoints. Depending on the model, these outputs may also include a **bounding box** around the detected person and one or more **confidence scores**.

### 2D vs 3D Estimates

These geometric predictions may be expressed in:
- **2D image/digital space**, where keypoints are expressed as pixel coordinates $(u,v)$ in the input image, or
- **3D world/metric space**, where keypoints are expressed as real metric spatial coordinates $(x,y,z)$ \[cm\] relative to the camera or some define origin in the real world.

**2D pose estimates** are often sufficient for tasks such as posture classification, simple activity recognition, and some . However, 2D pose does not directly describe depth, so different 3D body configurations can sometimes appear similar when projected into a 2D image and the size of a 2D pose is impact not only by the size of the person but also their distance from the camera.

**3D pose estimates** provides richer information about body orientation, limb depth, and movement in space, which can be more useful for biomechanical analysis and clinical interpretation. However, 3D HPE is usually more difficult and less reliable than 2D HPE because depth is harder to infer and suitable labelled datasets are much harder to collect.

<figure style="text-align: center;">
<img src="stereoscopic_3D_HPE.png" width="900" align="center">
<figcaption>Triangulation of 3D poses from two separate 2D estimates from a dual-camera stereoscopic system</figcaption>
</figure>

### Confidence Scoring

Many HPE systems produce not only geometric outputs, but also **confidence scores**. These scores help indicate how certain the model is about a predicted person or keypoint. However, “confidence” can mean different things depending on the model. Confidence scores are useful for filtering unreliable predictions during inference and for helping downstream systems decide how much trust to place in each estimate.

For the confidence score associated with a bounding box usually reflects how likely the model thinks it has detected a real person. In models that also perform multi-class object classification confidence is reflected in the predicted probability of the human class. 

For an individual keypoint, confidence may represent one of two related but different ideas:
1. **Localization confidence**: how strongly the model believes that the keypoint is located at a particular position.
2. **Visibility or presence confidence**: how likely the keypoint is to be visible or present in the image.

In heatmap-based HPE models (which we cover in the next section), keypoint confidence is often derived from the predicted heatmap itself. A stronger peak in the heatmap usually indicates greater confidence in the predicted keypoint location. In this case, the confidence score does not necessarily mean that the joint is visible; it may simply mean that the model has strong evidence for its estimated location.

In other models, especially direct-regression or detection-based pose models, confidence may be predicted explicitly as an extra output value for each keypoint. In this setting, the score may behave more like a learned probability that the keypoint is present, visible, or reliable.

It is important not to confuse these ideas. A model may be highly confident about the estimated location of a joint even if that joint is partly occluded, because the model can infer its position from surrounding body structure. Therefore, **keypoint confidence is not always the same as visibility**.

## Single-Person Monocular RGB-based Human Pose Estimation

For image-based HPE, computational cost scales roughly with the **number of input pixels**. To keep computation manageable and to standardise the input format, an image of arbitrary size 
$$I_{\text{orig}} \in \mathbb{R}^{H_{\text{orig}} \times W_{\text{orig}} \times 3}$$
is typically resized to a fixed input resolution 
$$I_{\text{in}} \in \mathbb{R}^{H_{\text{in}} \times W_{\text{in}} \times 3}$$
before being fed into the HPE model.

For 2D HPE, it is useful to distinguish between a **discrete image grid** and a **continuous coordinate space**. Using **0-based coordinates**, the discrete pixel lattice of the input image is 
$$\Lambda_{\text{in}} = \{0,\dots,W_{\text{in}}-1\} \times \{0,\dots,H_{\text{in}}-1\},$$
while the keypoints themselves are treated as continuous coordinates in the image plane: 
$$p_{\text{in},k} = (x_{\text{in},k}, y_{\text{in},k}) \in \Omega_{\text{in}},k=1,\dots,K,$$
where $K$ is the number of keypoints and $\Omega_{\text{in}}$ is the continuous input-image domain.

Depending on convention, coordinates may refer to either:

- **pixel centres**, where integer coordinates correspond to pixel centres, or
- **pixel corners**, where integer coordinates correspond to grid intersections at pixel boundaries.

For **3D HPE**, the keypoints are typically written as $p_{\text{in},k} = (x_k, y_k, z_k),$ and are usually defined in a **camera-relative**, **root-relative** (e.g. pelvis at the origin), or another chosen 3D reference frame.

<figure style="text-align: center;">
<img src="direct_vs_heatmap_regression.png" width="1000" align="center">
<figcaption>Direct regression vs heatmap regression for 2D single-person pose estimation</figcaption>
</figure>

### Direct Regression

Single-person pose estimation (SPPE) models based on direct regression usually assume that the input image contains **exactly one person** and directly predict that person’s pose.

* A 2D model may output $\hat{p} \in \mathbb{R}^{K \times 2}$
    * (or $\hat{p} \in \mathbb{R}^{K \times 3}$ if a confidence score is also predicted for each keypoint).
* A 3D model may output $\hat{p} \in \mathbb{R}^{K \times 3}$
    * (or $\hat{p} \in \mathbb{R}^{K \times 4}$ if a confidence score is also predicted).

Examples: [DeepPose](https://openaccess.thecvf.com/content_cvpr_2014/papers/Toshev_DeepPose_Human_Pose_2014_CVPR_paper.pdf), [Compositional Human Pose Regression](https://openaccess.thecvf.com/content_ICCV_2017/papers/Sun_Compositional_Human_Pose_ICCV_2017_paper.pdf)

These models are often fast and memory-efficient, which makes them attractive for lightweight or real-time applications. However, they are often less spatially precise than heatmap-based methods. Because they predict coordinates directly, they do **not** suffer from the same heatmap discretisation issue that arises when coordinates are decoded from low-resolution heatmaps.

#### Direct Regression Loss

Direct-regression models are usually trained using coordinate-space losses such as:
- **MAE / L1 loss**
- **MSE / L2 loss**
- or other losses defined directly between predicted joint coordinates $\hat{p}$ and $p$.

These losses supervise the final coordinates directly, but they provide **less explicit spatial guidance** because the model is not trained to produce a full spatial confidence map showing where evidence for each joint should lie in the image.

### Heatmap Regression

Heatmap-regression methods usually provide better localisation accuracy and can be more robust to ambiguity or partial occlusion. Instead of predicting a joint coordinate directly, the model predicts a **likelihood map** for each keypoint.

Examples: [RTMPose](https://arxiv.org/abs/2303.07399), [HRNet](https://openaccess.thecvf.com/content_CVPR_2019/papers/Sun_Deep_High-Resolution_Representation_Learning_for_Human_Pose_Estimation_CVPR_2019_paper.pdf) 

These models output tensors with shape,

$$ H_{\text{out}} \times W_{\text{out}} \times K$$

This allows the model to represent spatial uncertainty more naturally and to use local image structure to decide where each joint is most likely to be.

These models usually have higher computational and memory cost than direct-regression methods. Their outputs are also defined on a discrete heatmap grid, which can introduce **quantisation error** when converting heatmap peaks back into continuous image coordinates.

#### Coordinate Spaces in Heatmap Regression

For clarity, it helps to distinguish three related coordinate spaces:

1. **Original image space**, associated with the raw image $I_{\text{orig}}$
2. **Input image space**, associated with the resized image $I_{\text{in}}$
3. **Output / heatmap space**, associated with the model output

For simplicity, suppose the original image is resized to the input image by isotropic scaling with factor $s_{\text{in}}$, so that

$$
W_{\text{in}} = \frac{W_{\text{orig}}}{s_{\text{in}}}, \qquad
H_{\text{in}} = \frac{H_{\text{orig}}}{s_{\text{in}}}.
$$

Then an original-space keypoint
$$
p_{\text{orig},k} = (x_{\text{orig},k}, y_{\text{orig},k})
$$
maps into input space as

$$
p_{\text{in},k} =
\left(
\frac{x_{\text{orig},k}}{s_{\text{in}}},
\frac{y_{\text{orig},k}}{s_{\text{in}}}
\right).
$$

Next, suppose the network has an overall output stride $s_{\text{out}}$, so that the predicted heatmaps have spatial dimensions
$$
H_{\text{out}} = \frac{H_{\text{in}}}{s_{\text{out}}}, \qquad
W_{\text{out}} = \frac{W_{\text{in}}}{s_{\text{out}}}.
$$
Then the input-space keypoint maps into output/heatmap space as
$$
p_{\text{out},k} 
=
\left(
\frac{x_{\text{in},k}}{s_{\text{out}}},
\frac{y_{\text{in},k}}{s_{\text{out}}}
\right).
$$

Here, $x_{\text{out},k}$ and $y_{\text{out},k}$ are generally **continuous-valued** coordinates in heatmap space, so the true keypoint centre may lie **between** discrete heatmap pixels.

The discrete heatmap lattice is
$$
\Lambda_{\text{out}} = \{0,\dots,W_{\text{out}}-1\} \times \{0,\dots,H_{\text{out}}-1\},
$$
with pixel/cell indices $(u_{\text{out}},v_{\text{out}})$.

#### Ground-Truth Encoding

In 2D heatmap regression, the target for each keypoint is usually a 2D heatmap with a Gaussian blob centred at the true keypoint location in heatmap space. 

<figure style="text-align: center;">
<img src="heatmap_gt.jpeg" width="900" align="center">
<figcaption>Encoding ground truth for 2D heatmap regression</figcaption>
</figure>

A common target for keypoint $k$ is

$$
P_k(u_{\text{out}},v_{\text{out}}) = \exp\left(
-\frac{(u_{\text{out}} - x_{\text{out},k})^2 + (v_{\text{out}} - y_{\text{out},k})^2}{2\sigma^2}
\right),
$$

where $(u_{\text{out}},v_{\text{out}}) \in \Lambda_{\text{out}}$ and $\sigma$ controls the spread of the Gaussian.
This means that the Gaussian is evaluated only at the discrete heatmap pixels, even though its centre $(x_{\text{out},k},y_{\text{out},k})$ may be non-integer.

<figure style="text-align: center;">
<img src="gaussian_blob.png" width="500" align="center">
<figcaption>Gaussian blob evaluated around one pixel. Coordinates u,v refer to pixel corners.</figcaption>
</figure>

#### Prediction Decoding

A naive decoding approach is to take the brightest pixel in each predicted heatmap:
$$
\hat{p}_{\text{out},k}  = \arg\max_{(u_{\text{out}},v_{\text{out}})} \hat{P}_k(u_{\text{out}},v_{\text{out}}),
$$
and then map this discrete location back into the input image:
$$
\hat{p}_{\text{in},k} = s_{\text{out}} \, \hat{p}_{\text{out},k}.
$$

This gives one predicted coordinate per keypoint. However, because the prediction is restricted to the discrete heatmap grid, the decoded location is only an approximation to the true continuous coordinate. This is a key source of **quantisation error** in heatmap-based pose estimation.

More refined decoding methods, such as soft-argmax, integral regression, or local peak refinement, aim to reduce this error by estimating a sub-pixel coordinate from the local heatmap distribution.

<figure style="text-align: center;">
<img src="quantisation_error__.png" width="400" align="center">
</figure>


#### Extension to 3D Heatmap Regression

The same idea extends naturally to 3D pose estimation. Instead of defining keypoints in a continuous 2D image plane, we define them in a **bounded** continuous 3D space,

$$
\Omega_{\text{in}} \subset \mathbb{R}^3,
$$
for example a camera-relative or root-relative volume chosen to contain the person/scene/room. The ground-truth location of keypoint $k$ is then
$$
p_{\text{in},k} = (x_k, y_k, z_k) \in \Omega_{\text{in}}.
$$
This continuous 3D space is discretised into a voxel grid matching the spatial dimensions of the model output. If the output volume has dimensions $D_{\text{out}} \times H_{\text{out}} \times W_{\text{out}}$, then the discrete output lattice is
$$
\Lambda_{\text{out}} = \{0,\dots,W_{\text{out}}-1\} \times \{0,\dots,H_{\text{out}}-1\} \times \{0,\dots,D_{\text{out}}-1\}.
$$

Each keypoint is mapped from the continuous 3D space into a continuous coordinate in this discretised output space, and a 3D Gaussian target is centred at the voxel cell containing this keypoint. The model predicts one 3D heatmap per keypoint, and decoding selects or refines the highest-response voxel to recover the final 3D joint estimate.

Examples: [MeTRo](https://iliad-project.eu/wp-content/uploads/papers/Metric.pdf), [Coarse-to-Fine Volumetric Prediction for Single-Image 3D Human Pose](https://openaccess.thecvf.com/content_cvpr_2017/papers/Pavlakos_Coarse-To-Fine_Volumetric_Prediction_CVPR_2017_paper.pdf)

Full 3D heatmaps are expensive. Their cost grows with the size of the 3D volume, so memory and computation increase much faster than in 2D. For this reason, lower-resolution voxel grids are often used in practice, although this increases discretisation and quantisation error.

<figure style="text-align: center;">
<img src="3d_heatmap_regression.jpeg" width="450" align="center">
<figcaption>Heatmap regression for 3D pose estimation. A bounded continuous 3D space, for example 2.2x2.2x2.2m, is discretised into a voxel grid matching the dimensions of the model output.</figcaption>
</figure>


#### Refinements for Quantisation Error in Heatmap Regression

https://openaccess.thecvf.com/content_CVPR_2020/papers/Zhang_Distribution-Aware_Coordinate_Representation_for_Human_Pose_Estimation_CVPR_2020_paper.pdf

Several refinements have been proposed to reduce the limitations of basic heatmap regression, especially quantisation error and mismatch between the target heatmap and the true continuous keypoint location.

**1. Higher-resolution heatmaps**  
A simple mitigation is to predict heatmaps at higher spatial resolution. This reduces discretisation error because each heatmap pixel corresponds to a smaller region in the input image. However, it does not solve the problem entirely because it is still not practical for heatmaps to match the same resolution as the original image.

**2. Soft-argmax / integral regression**  
Another idea is to avoid hard argmax altogether. Instead of selecting the single brightest pixel, the coordinate can be estimated as a weighted average over the heatmap, where brighter locations contribute more. These methods still use heatmaps, but decode them into continuous coordinates more smoothly. For examle: [Integral Human Pose Regression](https://openaccess.thecvf.com/content_ECCV_2018/papers/Xiao_Sun_Integral_Human_Pose_ECCV_2018_paper.pdf), [Human Pose Regression by Combining Indirect Part Detection and Contextual Information](https://arxiv.org/pdf/1710.02322)

**3. Adaptive target encoding**  
Instead of using the same Gaussian width $\sigma$ for every keypoint in every image, some methods adjust the target distribution according to factors such as person scale, keypoint type, or localisation difficulty. This can make the supervision signal more appropriate for different joints and image conditions. For example: [Rethinking the Heatmap Regression for Bottom-up Human Pose Estimation](https://openaccess.thecvf.com/content/CVPR2021/papers/Luo_Rethinking_the_Heatmap_Regression_for_Bottom-Up_Human_Pose_Estimation_CVPR_2021_paper.pdf), [Motion-Aware Heatmap Regression for Human Pose Estimation in Videos](https://www.ijcai.org/proceedings/2024/0138.pdf)

**4. Continuous or grid-free representations**  
Some newer methods avoid relying entirely on a fixed discrete heatmap grid by representing keypoint likelihood in a more continuous way. These approaches are less common than standard heatmap methods, but they are motivated by the same problem: fixed low-resolution grids limit localisation precision. For example: [Continuous Heatmap Regression for Pose Estimation
via Implicit Neural Representation](https://proceedings.neurips.cc/paper_files/paper/2024/file/b90cb10d4dae058dd167388e76168c1b-Paper-Conference.pdf)

#### Heatmap Regression Loss

Heatmap-regression methods are trained by supervising the **spatial evidence map** for each keypoint (in output/heatmap space). This gives the model more explicit spatial guidance than direct coordinate regression, because it learns not only the final keypoint position but also where evidence for that keypoint should appear in the image.

Common loss functions include:

- **Mean squared error (MSE / L2 loss)**  
  This is the most common choice. The predicted heatmap for each joint is compared with the target heatmap centred at the ground-truth location in output space.  
$$
\mathcal{L}_{\text{MSE}}
  =
  \frac{1}{KW_{\text{out}}H_{\text{out}}}
  \sum_{k=1}^{K}
  \sum_{u_{\text{out}}=0}^{W_{\text{out}}-1}\sum_{v_{\text{out}}=0}^{H_{\text{out}}-1}
  \left(\hat{P}_k(u_{\text{out}},v_{\text{out}})-P_k(u_{\text{out}},v_{\text{out}})\right)^2.
$$
- **Cross-entropy over a discrete location**  
  Instead of using a Gaussian target, the true keypoint location can be represented as a discrete target on the heatmap grid, and the model is trained using cross-entropy over spatial positions.  Example: [Mask R-CNN](https://openaccess.thecvf.com/content_ICCV_2017/papers/He_Mask_R-CNN_ICCV_2017_paper.pdf)

- **Binary cross-entropy (BCE)**  
  Each heatmap location is treated as a binary prediction problem, indicating whether that location belongs to the target keypoint or not. This is less common than MSE, but it is still a valid way to supervise heatmaps.

### BlazePose

An SPPE model which gets the best of both direct and heatmap regression. It has 2 output heads. One performs direct regression and is used during training and testing. One performs heatmap regression but is only used during training. Test-time inference only considers outputs from direct regression. Backpropagation of the heatmap regression loss helps the earlier convolutional layers to produce higher quality features which could not be produced from the direct regression learning objective. When the model is deployed, the layers necessary for heatmap regression are removed and the remaining layers directly regress landmark coordinates.

Further reading on BlazePose:
* https://arxiv.org/pdf/2006.10204ere
* https://research.google/blog/on-device-real-time-body-pose-tracking-with-mediapipe-blazepose/
* https://www.emergentmind.com/topics/blazepose

### SPPE Evaluation

Simple/intuitive single-person evaluation metrics measure the Euclidean distance between each predicted and ground-truth keypoints in original/input space:
$$
d_k = \left\lVert \hat{p}_k - p_k \right\rVert_2.
$$
where:
- $\hat{p}_k$ is the predicted location of keypoint $k$,
- $p_k$ is the ground-truth location,

In **3D pose estimation**, this is often meaningful directly because keypoints are expressed in a physical or camera-relative 3D coordinate system. This gives rise to metrics such as:

- **JPE**: Joint Position Error for a single joint
- **MPJPE**: Mean Per-Joint Position Error, i.e. the average Euclidean distance across all joints

$$
\text{MPJPE} = \frac{1}{K}\sum_{k=1}^{K} \left\lVert \hat{p}_k - p_k \right\rVert_2.
$$

In **2D image space**, however, raw Euclidean distance in pixels is **not scale-invariant**. For example, the same 10-pixel error is much less serious for a large pose close to the camera than for a small pose far away. For this reason, 2D SPPE usually uses **normalised distance-based metrics** rather than raw pixel error.

#### PCK — Percentage of Correct Keypoints

A predicted keypoint is counted as **correct** if its distance from the ground-truth keypoint is below some threshold proportional to a reference body scale:

$$
\text{PCK}@\alpha = \frac{1}{K}\sum_{k=1}^{K} \mathbb{1}\!\left(
\frac{\left\lVert \hat{p}_k - p_k \right\rVert_2}{s} < \alpha
\right),
$$

where:
- $\hat{p}_k$ is the predicted location of keypoint $k$,
- $p_k$ is the ground-truth location,
- $s$ is a reference scale for the person or body,
- $\alpha$ is a chosen threshold,
- $\mathbb{1}(\cdot)$ is an indicator function equal to 1 if the condition is true, else 0.

#### PCKh — Percentage of Correct Keypoints normalised by head size

PCKh is a specific version of PCK in which the reference scale is the person’s **head size**. In the [MPII dataset](https://openaccess.thecvf.com/content_cvpr_2014/papers/Andriluka_2D_Human_Pose_2014_CVPR_paper.pdf), annotations include bounding boxes around people's heads. They define the diagonal length of the box $d_{head}$ and then define head scale as 60% of this length $s_{head}=0.6 d_{head}$

#### PCK AUC - Area Under the PCK Curve

PCK depends on the threshold $\alpha$. A loose threshold makes the model look better, while a strict threshold makes the evaluation harder. To reduce dependence on any single threshold, we can vary $\alpha \in [\alpha_{min}, \alpha_{max}]$, plot the resulting PCK curve (PCK vs $\alpha$), and compute the **area under the curve (AUC)**.

$$
\text{AUC} = \int_{\alpha_{\min}}^{\alpha_{\max}} \text{PCK}(\alpha)\, d\alpha.
$$

Where larger AUC indicate better performance. In practice, this is computed numerically over a finite set of thresholds. AUC measures how well the model performs **across a range of tolerances**, rather than at just one chosen tolerance. It provides a more complete view of localisation quality and makes the evaluation less sensitive to one arbitrary threshold.

This is not to be confused with **ROC AUC** (Area Under the Receiver Operating Characteristic Curve) which is a common evaluation metric used in general classification tasks. This involves plotting True Positive Rate vs False Positive Rate as a positive detection threshold is varied.

<figure style="text-align: center;">
<img src="PCK_AUC_example.jpeg" width="600" align="center">
<figcaption>Example of the PCK AUC used to compare several pose estimation models from the paper 'Vision-Based Pose Forecasting of Construction Equipment for Monitoring Construction Site Safety'</figcaption>
</figure>

#### IoU: Intersection Over Union

Standard metric used to measure how well a predicted bounding box overlaps with a ground-truth bounding box. It is widely used in object detection.

Given a predicted bounding box $B_{pred}$ and a ground-truth bounding box $B_{gt}$, IoU is defined as

$$ IoU = \frac{| \hat{B} \cap B |}{| \hat{B} \cup B |}$$

where:
* $| \hat{B} \cap B |$ is the area of overlap between the two boxes,
* $| \hat{B} \cup B |$ is the total area covered by either box.

<figure style="text-align: center;">
<img src="IoU.jpeg" width="600" align="center">
</figure>

#### OKS: Object Keypoint Similarity

OKS is a keypoint analogue of IoU for pose estimation. It measures how well a predicted pose matches a ground-truth pose while accounting for:
- person scale
- keypoint type
- keypoint visibility
  
$$ OKS = \frac{\sum_{k=1}^{K}{\exp(-\frac{d_k^2}{2a^2r_k^2})\delta (v_k >0))}}{\sum_{k=1}^{K}{\delta (v_n > 0)}} $$

where $d_k$ is the Euclidean distance between the $k$th detected keypoint and corresponding ground-truth keypoint, and $a$ is the area of the ground-truth bounding box. The term $r_k$ is the per-keypoint falloff constant that sets how quickly the OKS decays with localisation error. It is estimated from images which have been labelled by multiple people and measures the annotator-to-annotator spread for each keypoint type on the same person after normalising by person size. $r_k$ is then set to the standard deviation of this normalised difference. Lastly, $v_k$ indicates the visibility of the $k$th keypoint, where $\delta (v_n > 0)=1$ if the keypoint is visible, else 0.

Instead of declaring a joint simply correct or incorrect, OKS gives a **soft similarity score** between 0 and 1. Small localisation errors produce scores close to 1, while larger errors are penalised more strongly.

<figure style="text-align: center;">
<img src="OKS.jpeg" width="600" align="center">
<figcaption>Demonstration of the Object Keypoint Similarity. Euclidean distance between prediction (red) and ground 
truth position (blue) of the eye is equal to that of the wrist (green). However, the the per-keypoint falloff
constan r_k is equal to 0.85 for the wrist and 0.5 for eye, resulting in a lower keypoint similarity for eye to 
account for the fact that this is a much smaller anatomical surface than the wrist and therefore labl 
variability is higher. </figcaption>
</figure>

# Multi-Person Human Pose Estimation

SPPE is not sufficient for most applications in remote patient monitoring since we cannot assume that there will only ever be a single person in the frame. Even for telerehabiliation and point-in-time assessments, a carer could be present to assist and an SPPE may sporadically switch between estimating the pose for the patient and carer. Multi-person pose estimation (MPPE) is more suitable. 

Multi-person pose estimation (MPPE) is more difficult because it combines two problems at once: **pose localisation** and **instance separation**. The model must not only estimate where keypoints are, but also determine which keypoints belong to the same person.

A useful guiding question is:
**How can a model represent multiple candidate detections within a fixed-size output structure?**

In this section, we restrict discussion to **2D MPPE**.

<figure style="text-align: center;">
<img src="top_down_bottom_up.jpeg" width="900" align="center">
<figcaption>.</figcaption>
</figure>

## Two-Stage (Top-Down) MPPE

Top-down MPPE first applies a **person detector** to identify and localise multiple people, usually by estimating their bounding boxes. The image is then cropped around each detected person, and each crop is passed independently to an SPPE model.

This strategy simplifies pose estimation because each SPPE only needs to solve the easier problem of estimating the pose of one person at a time. As a result, top-down methods often achieve strong accuracy when person detection is reliable.

**Advantages**
- reduces MPPE to a sequence of easier SPPE problems
- often achieves strong accuracy when people are clearly detectable
- benefits directly from advances in person detection and SPPE

**Weaknesses**
- performance is bottlenecked by the person detector
- missed or poor bounding boxes cause pose failures
- overlapping people and heavy occlusion can make detection unreliable
- repeated cropping and per-person inference increase computation
- end-to-end training is not easy

Examples: [AlphaPose](https://doi.org/10.1109/TPAMI.2022.3222784), [HRNet](https://openaccess.thecvf.com/content_CVPR_2019/papers/Sun_Deep_High-Resolution_Representation_Learning_for_Human_Pose_Estimation_CVPR_2019_paper.pdf) 

Some top-down methods further improve the detector-to-SPPE pipeline by refining bounding boxes, incorporating global context, reducing crop-and-resize distortions, or explicitly handling occlusion. For example: [RMPE](https://openaccess.thecvf.com/content_ICCV_2017/papers/Fang_RMPE_Regional_Multi-Person_ICCV_2017_paper.pdf)

## Single-Stage MPPE

Single-stage MPPE predicts multiple people in **one forward pass** through a single end-to-end model. Instead of first detecting people and then processing each detected region separately, the model produces multi-person predictions directly from the full image.

### Bottom-Up MPPE

Bottom-up methods first predict all keypoints jointly over the whole image, without initially assigning them to particular people. A separate grouping step is then required to associate the detected keypoints into individual person instances.

The main challenge in bottom-up MPPE is therefore not only keypoint localisation, but also **association**.

**Advantages**
- processes the full image only once
- can scale better when many people are present
- avoids dependence on a separate person detector

**Weaknesses**
- grouping keypoints into the correct people is difficult
- association becomes especially challenging in crowded and occluded scenes

Examples: [OpenPose](https://openaccess.thecvf.com/content_cvpr_2017/papers/Cao_Realtime_Multi-Person_2D_CVPR_2017_paper.pdf), [Associate Embedding](https://proceedings.neurips.cc/paper_files/paper/2017/file/8edd72158ccd2a879f79cb2538568fdc-Paper.pdf)


**OpenPose**

OpenPose predicts:
- a set of **per-keypoint heatmaps**, indicating likely locations of each joint type. This uses heatmap-regression, though unlike SPPE, each keypoint heatmap can have multiple peaks. 
- a set of **per-limb Part Affinity Fields (PAFs)**, which are maps which encode whether pairs of keypoints belong to the same limb. 

Each cell in a PAF stores a 2D vector. For pixels lying on a particular limb, this vector points from one joint toward the other joint of that limb. These vector fields provide information about both limb location and limb orientation.

At inference time, candidate keypoints are first extracted from the heatmaps. The PAFs are then used to score possible connections between keypoints, allowing the model to assemble detected joints into full person skeletons.

<figure style="text-align: center;">
<img src="OpenPose_output.jpeg" width="900" align="center">
<figcaption></figcaption>
</figure>

### Direct Instace-Based MPPE

Direct Instace-Based MPPE methods also predict multiple people in one forward pass, but unlike bottom-up methods, they aim to produce keypoints in a form which implicitly associates keypoints to person instances. In other words, they reduce or avoid the need for an explicit post hoc grouping stage.

These methods are often inspired by ideas from modern object detection, such as dense multi-scale prediction or direct instance prediction.

**Example: YOLO-Pose** predicts candidate person instances directly from multi-scale feature maps. Each candidate detection includes not only bounding-box-related information, but also the corresponding keypoint predictions for that person. The final set of person poses is then obtained by filtering low-confidence and duplicate candidate detections.

### YOLO-Pose

You Only Look Once (YOLO) refers to a family of object-detection models designed around a simple idea: instead of first extracting many candidate image regions and then processing each region separately, the model processes the entire image once and produces a large set of candidate detections in parallel. This is why YOLO is closely associated with real-time performance. The model does not “search” the image one object at a time. Rather, it constructs a hierarchy of feature maps and then asks, at many spatial locations and scales simultaneously, whether an object is present and, if so, where it is.

YOLO-Pose is a family of pose estimation models which extend YOLO for pose estimation by introducing additional task-specific detection heads.

[A Comprehensive Review of Yolo Architectures in Computer Vision: From Yolov1 to Yolov8 and Yolo-Nas](https://arxiv.org/pdf/2304.00501)

[YOLO-Pose: Enhancing YOLO for Multi Person Pose Estimation Using Object Keypoint Similarity Loss](https://openaccess.thecvf.com/content/CVPR2022W/ECV/papers/Maji_YOLO-Pose_Enhancing_YOLO_for_Multi_Person_Pose_Estimation_Using_Object_CVPRW_2022_paper.pdf)

#### YOLO-Pose Architecture
A useful way to understand YOLO is through the common backbone–neck–head decomposition.

##### Backbone
The **backbone** is the main feature extractor. It receives the input image and transforms it through a sequence of convolutional layers into a hierarchy of increasingly abstract feature maps. Suppose the input image has been reshaped to a fixed input space of
$$ H_{\text{in}} \times W_{\text{in}} = 640 \times 640 $$
In the earlier layers of the backbone, the feature maps remain relatively high resolution, so they retain fine spatial detail such as edges, corners, contours, and local textures. In the deeper layers, repeated downsampling reduces the spatial dimensions but increases the receptive field and semantic abstraction. These deeper features are less precise spatially, but they are much better at representing larger object parts, object identity, and scene context.

Modern YOLO models are designed to preserve and exploit this multi-scale hierarchy. In a four-scale design, the model works with feature maps associated with output strides
$$
s_{\text{out},i} \in \{8,16,32,64\}, i \in \{1,2,3,4\}
$$

The corresponding output-space dimensions are
$$
H_{\text{out},i} = \frac{H_{\text{in}}}{s_{\text{out},i}}, W_{\text{out},i} = \frac{W_{\text{in}}}{s_{\text{out},i}}
$$
For a $640 \times 640$ input, this gives:
* **P3** with $s_{\text{out},i}=8,$ so $H_{\text{out},1} \times W_{\text{out},1} = 80 \times 80$ 
* **P4** with $s_{\text{out},i}=16,$ so $H_{\text{out},2} \times W_{\text{out},2} = 40 \times 40$ 
* **P5** with $s_{\text{out},i}=32,$ so $H_{\text{out},3} \times W_{\text{out},3} = 20 \times 20$ 
* **P6** with $s_{\text{out},i}=64,$ so $H_{\text{out},4} \times W_{\text{out},4} = 10 \times 10$ 

These scales play different roles. **P3** has the finest spatial resolution, so it is most useful for localising smaller objects. **P6** has the coarsest spatial resolution, but the deepest semantic features and largest receptive field, so it is better suited to larger objects.
##### Neck
The **neck** sits between the backbone and the final prediction head. Its job is not simply to pass features onward, but to fuse, redistribute, and refine information across scales. This is important because the backbone alone produces a hierarchy in which shallow features are spatially precise but semantically weak, while deep features are semantically strong but spatially coarse. The neck attempts to combine the best of both. In practice, this usually involves passing high-level semantic information back down toward higher-resolution feature maps (top-down path), while also allowing lower-level localisation detail to influence coarser maps (bottom-up path). Top-down/bottom-up paths are not related to top-down/bottom-up approaches to MPPE, they just happen to share the same names. Conceptually, the neck helps ensure that each prediction scale has both:
1. enough spatial detail to localise objects accurately, and
2. enough semantic context to decide what the object is.

After this fusion process, the model outputs the refined multi-scale feature maps P3, P4, P5, and P6 which are then fed into the prediction head.

##### Head
The **head** converts these refined feature maps into candidate detections. For each scale $i$, there are multiple heads which specialise in different tasks, like object classification $\hat{P}^{cls}_{\text{out},i}$, object localisation $\hat{P}^{box}_{\text{out},i}$, pose estimation $\hat{P}^{pose}_{\text{out},i}$. The final outputs do not represent heatmaps. The key idea is that each spatial cell in each output map corresponds to a location in input space and is asked to predict whether an object is present there, and assuming an object does exist there, what does the bounding box look like, what kind of object is it, and assuming it is a human, what does the pose look like?

<figure style="text-align: center;">
<img src="YOLO_architecture.png" width="900" align="center">
<figcaption>Architecture of YOLO-Pose.</figcaption>
</figure>

##### YOLO Object Localisation

For each scale $i$, the output tensor for the object localisation head has the shape
$$H_{\text{out},i} \times W_{\text{out},i} \times 5.$$
Each cell in this map produces one candidate bounding box detection $\hat{P}^{box}_{\text{out},i}(u_{\text{out},i},v_{\text{out},i})$ which is a vector containing:
* $\hat{b_c}$: confidence score for the box, how confident the model is that the box contains and object and how accurate the box is.
* $\hat{t_x}, \hat{t_y}$: centers of the box relative to the grid cell $(u_{\text{out},i},v_{\text{out},i})$.
* $\hat{b_w}, \hat{b_h}$: height and width of the box relative to the full image in input space.

$\hat{b_c}, \hat{t_x}, \hat{t_y}, \hat{b_w}, \hat{b_h} \in (0,1)$, made possible using the sigmoid activation function. 

For a cell $(u_{\text{out},i},v_{\text{out},i})$ the predicted center location of the candidate bounding box in output space is $(\hat{b_x},\hat{b_y})=(u_{\text{out},i} + \hat{t_x},v_{\text{out},i} + \hat{t_y})$.

<figure style="text-align: center;">
<img src="YOLO_box_head_output.png" width="600" align="center">
<figcaption>How the output from YOLO's object localisation (for some scale i) can be used to decode a candidate bounding box detection.</figcaption>
</figure>

##### YOLO Object Classification

For each scale $i$, the output tensor for the object classification head has the shape
$$H_{\text{out},i} \times W_{\text{out},i} \times M.$$
Where $M$ is the number of object classes. Each cell in this map produces one classification vector of length $M$ where softmax is applied to ensure each cell is a probability distribution over the classes 
$$softmax(\hat{P}^{cls}_{\text{out},i}(u_{\text{out},i},v_{\text{out},i}))$$

<figure style="text-align: center;">
<img src="YOLOv1_output.png" width="600" align="center">
<figcaption>YOLO target labels.The figure depicts a simplified YOLO model with three-by-three output space and three classes. </figcaption>
</figure>

##### YOLO Pose Estimation

For each scale $i$, the output tensor for the pose estimation head has the shape
$$H_{\text{out},i} \times W_{\text{out},i} \times (2 \times K).$$
Where $K$ is the total number of keypoints in each pose. Each cell in this map produces one candidate pose estimate $\hat{P}^{pose}_{\text{out},i}(u_{\text{out},i},v_{\text{out},i})$ which is a vector containing the locations of all keypoints as offsets relative to the detected bounding box or cell.


### Non-Maximal Suppression

MPPE models (and object detection models more broadly) generate multiple detections with different confidence scores. It is common in post-processing to filter out incorrect detections by applying a threshold on confidence scores. But, often these models will confidently produce duplicate detections.

Non-maximum suppression (NMS) is a post-processing technique used in object detection and MPPE to reduce the number of duplicate candidate detections / pose estimates that correspond to the same object or person. The following algorithm describes the NMS procedure applied in object detection. 

<figure style="text-align: center;">
<img src="NMS_algo.jpeg" width="600" align="center">
<figcaption>NMS algorithm for object detection.</figcaption>
</figure>

<figure style="text-align: center;">
<img src="NMS_fig.jpeg" width="600" align="center">
<figcaption>NMS applied in object detection.</figcaption>
</figure>

NMS ranks all candidate detections by confidence, then iteratively retains the highest-confidence detections while removing any remaining detections where the IoU or OKS is above a predefined threshold. This process is repeated until no candidates remain. 

The same procedure could be applied directly to pose estimates by substituting the OKS with the IoU score. 

#### YOLO Loss

Loss for one detection head scale $i$:
$$L_{i} = \lambda_{\text{obj}} L_{\text{obj},i}+\lambda_{\text{cls}} L_{\text{cls},i}+\lambda_{\text{box}} L_{\text{box},i}+\lambda_{\text{pose}} L_{\text{pose},i}$$

The $\lambda$ values are tunable hyperparameters.

The final loss used in backpropagation:
$$L_{\text{total}}
=
L_1 + L_2 + L_3 + L_4$$

##### Objectness

$$L_{\text{obj},i}=\sum_{j=1}^{H_{\text{out},i} \times W_{\text{out},i}} \operatorname{BCE}(b_{c,j,i}, \hat{b}_{c,j,i})$$

* $\operatorname{BCE}$: binary cross-entropy loss
* $\hat{b}_{c,j,i}$: confidence score for the detection (objectness) at cell $j$ and scale $i$
* $b_{c,j,i} \in \{0,1\}$: ground truth objectness at cell $j$ and scale $i$. 

Assignment of cells to ground truth objects/people is a whole other topic. For now we can just assume a naive approach used in older versions of YOLO where objects/people are assigned to cells if they reside within the bounding box. 

##### Foreground Indicator

Older versions of YOLO use an indicator function $\mathbb{1}^{obj}(.)$ to ensure that we only packpropagate classification/box/pose loss for cells associated with an object/person (TP or FN detection). 

* $\mathbb{1}^{obj}_{j,i} = 1$ if cell $j$ at scale $i$ is associated with a object/person.
* $\mathbb{1}^{obj}_{j,i} = 0$ otherwise.

##### Classification
$$L_{\text{cls},i}=\sum_{j=1}^{H_{\text{out},i} \times W_{\text{out},i}} \mathbb{1}^{obj}_{j,i} \, \operatorname{CE}(P^{cls}_{\text{out},j,i}, \hat{P}^{cls}_{\text{out},j,i})$$

* $\operatorname{CE}$: cross-entropy loss
* $\hat{P}^{cls}_{\text{out},j,i}$: predicted class probabilities/logits for cell $j$ and scale $i$ (output from the cls head)
* $P^{cls}_{\text{out},j,i}$: ground-truth class label (one-hot class vector) for cell $j$ and scale $i$

##### Box Regression

$$L_{\text{box},i}=\sum_{j=1}^{H_{\text{out},i} \times W_{\text{out},i}} \mathbb{1}^{obj}_{j,i} \left(1 - \operatorname{IoU}(P^{box}_{out,j,i}, \hat{P}^{box}_{out,j,i})\right)$$

* $\operatorname{IoU}$: intersection over union between predicted and ground-truth boxes.
* $\hat{P}^{box}_{out,j,i}$: predicted bounding box for cell $j$ and scale $i$. (IoU ignores $\hat{b}_c$ term)
* $P^{box}_{out,j,i}$: ground-truth bounding box for cell $j$ and scale $i$. (IoU ignores $b_c$ term)

##### Pose/Keypoint


$$L_{\text{pose},i}=\sum_{j=1}^{H_{\text{out},i} \times W_{\text{out},i}} \mathbb{1}^{obj}_{j,i} \operatorname{OKS}(P^{pose}_{out,j,i}, \hat{P}^{pose}_{out,j,i})$$

* $\operatorname{OKS}$: Object keypoint similiarity.
* $\hat{P}^{pose}_{out,j,i}$: predicted pose for cell $j$ and scale $i$.
* $P^{pose}_{out,j,i}$: ground-truth pose for cell $j$ and scale $i$.